# SAGE — Multi-Crop Vision AI Inference API (Kaggle Deployment)
This notebook deploys the SAGE FastAPI backend on a Kaggle GPU instance (Tesla T4 or P100),
loads the **Qwen/Qwen2.5-VL-3B-Instruct** base model, dynamically attaches crop-specific LoRA adapters
from **IDK-Chaitanya/SAGE5Class-cp**, and securely exposes the REST API via a Cloudflare Tunnel.

### Step 1: Install Dependencies
Install pinned production backend requirements and tunneling utilities.

In [ ]:
!pip install -q -r requirements.txt
# Download cloudflared Linux binary for external tunneling
!curl -s -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

### Step 2: Repository and Workspace Setup
Ensure repository path is in `sys.path` and check current directory.

In [ ]:
import os
import sys

# If running from /kaggle/working and repo was cloned:
# !git clone https://github.com/YOUR_USERNAME/Sage-backend.git
# %cd Sage-backend

REPO_DIR = os.getcwd()
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"Workspace root: {REPO_DIR}")
!ls -la

### Step 3: Configure Hugging Face Authentication
Read Hugging Face token securely using Kaggle Secrets (`HF_TOKEN`).

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    print("✅ HF_TOKEN successfully loaded from Kaggle Secrets!")
except Exception as e:
    print("⚠️ Kaggle Secrets HF_TOKEN not found, checking environment:", e)
    if "HF_TOKEN" in os.environ:
        print("✅ HF_TOKEN present in environment.")
    else:
        print("❗ Please set HF_TOKEN in Kaggle Secrets (Add-ons -> Secrets).")

### Step 4: Verify GPU Availability and CUDA Specifications

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
    print(f"GPU Device: {device_name} ({vram_gb} GB VRAM)")
    print(f"BF16 Support: {torch.cuda.is_bf16_supported()}")
else:
    print("⚠️ WARNING: No GPU detected! Go to Settings -> Accelerator -> GPU T4 x2 or P100.")

!nvidia-smi

### Step 5: Verify Hugging Face Repository & Checkpoints
Run the checkpoint verification utility to validate all 5 crop adapters.

In [ ]:
!python scripts/verify_checkpoints.py

### Step 6: Inspect Checkpoint Files and Adapter Configurations

In [ ]:
from app.models.crop_registry import crop_registry
from app.models.adapter_loader import AdapterLoader

loader = AdapterLoader()
for crop_id in crop_registry.list_crop_ids():
    cfg = crop_registry.get_crop(crop_id)
    print(f"\nInspecting {cfg.display_name} ({crop_id})...")
    local_path = loader.download_checkpoint(cfg)
    meta = loader.inspect_checkpoint(crop_id, local_path)
    print(f"  is_lora: {meta.is_lora}")
    print(f"  rank: {meta.lora_rank}, alpha: {meta.lora_alpha}")
    print(f"  base_ref: {meta.base_model_reference}")

### Step 7: Pre-load Base Vision-Language Model
Load `Qwen/Qwen2.5-VL-3B-Instruct` into GPU memory.

In [ ]:
from app.models.model_manager import get_model_manager

manager = get_model_manager()
print("Loading base model into GPU...")
manager.load_base_model()
status = manager.get_model_status()
print("Base model loaded:", status.base_model_loaded)
print("GPU Memory Status:", status.gpu_status.model_dump())

### Step 8: Load Initial Crop Adapter (e.g., Tomato)

In [ ]:
tomato_crop = crop_registry.get_crop("tomato")
print(f"Attaching adapter for {tomato_crop.display_name}...")
manager.switch_adapter(tomato_crop)
status = manager.get_model_status()
print("Active Crop:", status.active_crop)
print("Active Adapter:", status.active_adapter)

### Step 9: In-Notebook Smoke Test with a Sample Image

In [ ]:
import io
from PIL import Image, ImageDraw
from app.inference.predictor import get_predictor

# Generate synthetic sample leaf image for smoke testing
test_img = Image.new("RGB", (400, 400), color=(34, 139, 34))
draw = ImageDraw.Draw(test_img)
draw.ellipse([150, 150, 250, 250], fill=(139, 69, 19))
buf = io.BytesIO()
test_img.save(buf, format="JPEG")
sample_bytes = buf.getvalue()

predictor = get_predictor()
custom_prompt = "Identify the disease affecting this tomato leaf and explain the visible symptoms."
print("Running local multimodal prediction smoke test on tomato with custom prompt...")
res = predictor.predict(crop_id="tomato", image_bytes=sample_bytes, prompt=custom_prompt)
print("Success:", res.success)
print("Active Adapter:", res.active_adapter)
print("Input Prompt:", res.input.prompt)
print("Inference Latency:", res.inference_time_seconds, "seconds")
print("\nModel Response:\n", res.prediction.response)


### Step 10: Start FastAPI Server in Background Process

In [ ]:
import subprocess
import time
import requests

# Start uvicorn server in background
server_proc = subprocess.Popen(
    ["python", "scripts/start_server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(f"Server process started with PID: {server_proc.pid}")

# Wait for server to become healthy
for attempt in range(15):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.status_code == 200:
            print("✅ SAGE FastAPI server is LIVE:", r.json())
            break
    except Exception:
        time.sleep(2)
else:
    print("❗ Server health check timed out. Check server logs below:")
    stdout, _ = server_proc.communicate(timeout=5)
    print(stdout)

### Step 11: Expose Server Externally via Cloudflare Tunnel
Cloudflare Tunnel provides an encrypted, high-speed public HTTPS URL with no sign-up or open ports required.
> **Note:** Kaggle allocates dynamic IP environments. When the notebook session restarts, a fresh tunnel URL will be assigned.

In [ ]:
import subprocess
import time
import re

# Launch Cloudflare Tunnel in background
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
print("Establishing Cloudflare Tunnel...")
for _ in range(25):
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(1)
        continue
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("=" * 60)
    print(f"🚀 PUBLIC API URL: {public_url}")
    print(f"📖 SWAGGER DOCS:  {public_url}/docs")
    print("=" * 60)
else:
    print("Failed to extract tunnel URL. Please check cloudflared output.")

### Step 12: Test External Public API Endpoints via curl & requests

In [ ]:
import requests

if public_url:
    # 1. Health check
    h_res = requests.get(f"{public_url}/health")
    print("Health Check Status:", h_res.status_code, h_res.json())

    # 2. Available Crops check
    c_res = requests.get(f"{public_url}/api/v1/crops")
    print("Available Crops:", c_res.json().get("available_crops"))

    # 3. Multimodal predict endpoint test (crop + image + prompt)
    files = {"image": ("test_leaf.jpg", sample_bytes, "image/jpeg")}
    data = {
        "crop": "tomato",
        "prompt": "Identify the disease affecting this tomato leaf and explain the visible symptoms."
    }
    p_res = requests.post(f"{public_url}/api/v1/predict", data=data, files=files)
    print("\n--- Multimodal Tomato Disease Prediction Result ---")
    print("Status Code:", p_res.status_code)
    print(p_res.json())
else:
    print("Please provide a running tunnel URL to test external endpoints.")
